# eval_dataset_builder

Builds **candidate pools** and **LLM labels** for multiple chapter specs.

Outputs:
- `eval_dataset/blueprints/<chapter_id>.json`
- `eval_dataset/fetch/openalex_<chapter_id>.csv` and `eval_dataset/fetch/semantic_scholar_<chapter_id>.csv`
- `eval_dataset/stageA/stageA_combined_oa_s2_<chapter_id>.csv`
- `eval_dataset/candidates/*.csv`
- `eval_dataset/labels/*.csv`
- `eval_dataset/labeled_dataset.csv`


In [3]:
import os
import re
import json
import time
import random
import asyncio
import hashlib
from pathlib import Path
from datetime import datetime, timedelta, timezone
from typing import List, Optional, Literal, Dict, Any, Iterable

import requests
import numpy as np
import pandas as pd

from pydantic import BaseModel, Field
from tqdm.auto import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from openai import OpenAI
from agents import Agent, Runner, ModelSettings

# -----------------------------
# Config
# -----------------------------
SEED = 42
np.random.seed(SEED)

OUT_DIR = Path("eval_dataset")
BLUEPRINT_DIR = OUT_DIR / "blueprints"
CAND_DIR = OUT_DIR / "candidates"
LABEL_DIR = OUT_DIR / "labels"
FETCH_DIR = OUT_DIR / "fetch"
STAGEA_DIR = OUT_DIR / "stageA"
for d in [OUT_DIR, BLUEPRINT_DIR, CAND_DIR, LABEL_DIR, FETCH_DIR, STAGEA_DIR]:
    d.mkdir(exist_ok=True)

STAGEA_ALL_CSV = STAGEA_DIR / "stageA_all_chapters.csv"

# Local caches
EMBED_CACHE_DIR = Path(".embed_cache")
EMBED_CACHE_DIR.mkdir(exist_ok=True)

LABEL_CACHE_DIR = Path(".llm_label_cache_v1")
LABEL_CACHE_DIR.mkdir(exist_ok=True)

# -----------------------------
# API keys
# -----------------------------
# OpenAI (for blueprint + labeling)
assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY env var."
client = OpenAI()

# OpenAlex (recommended)
OPENALEX_API_KEY = os.getenv("OPENALEX_API_KEY", "").strip()
if not OPENALEX_API_KEY:
    print("Warning: OPENALEX_API_KEY not set; continuing without it (may be slower / more rate limits).")

# Semantic Scholar (recommended)
S2_API_KEY = os.getenv("S2_API_KEY", "").strip()

# -----------------------------
# Model selection
# -----------------------------
# Blueprint: only 3 calls => ok to use a stronger model.
BLUEPRINT_MODEL = "gpt-5-mini"

# Labeling: keep it cheap; optional pass2 on uncertain examples.
LABEL_MODEL_PASS1 = "gpt-5-nano"
LABEL_MODEL_PASS2 = "gpt-5-mini"
USE_PASS2 = True

# Pass2 selection rules
PASS2_CONFIDENCE_LT = 70
PASS2_REDO_MAYBE = True
PASS2_RANDOM_AUDIT_PER_CHAPTER = 10

# -----------------------------
# Budget + pricing (USD)
# -----------------------------
# Source: OpenAI pricing pages (update here if pricing changes).
MODEL_PRICES_USD_PER_1M = {
    # text models
    "gpt-5-nano": {"input": 0.05, "cached": 0.005, "output": 0.40},
    "gpt-5-mini": {"input": 0.25, "cached": 0.025, "output": 2.00},
    # embeddings
    "text-embedding-3-small": {"input": 0.02, "cached": 0.0, "output": 0.0},
}

BUDGET_USD = 2.0
STOP_ON_BUDGET = True

# -----------------------------
# Stage A (API fetching) knobs
# -----------------------------
FETCH_FORCE = False
RUN_OPENALEX = True
RUN_SEMANTIC_SCHOLAR = True

FETCH_MAX_QUERIES_PER_CHAPTER = 15  # main_query + up to ~14 facets

# OpenAlex
OA_BASE_URL = "https://api.openalex.org/works"
OA_PER_PAGE = 100
OA_MAX_WORKS_PER_QUERY = 300
OA_TIMEOUT_SEC = 30

# Semantic Scholar
S2_BASE = "https://api.semanticscholar.org/graph/v1"
S2_SEARCH_URL = f"{S2_BASE}/paper/search"
S2_BATCH_URL  = f"{S2_BASE}/paper/batch"

S2_LIMIT = 100
S2_MAX_PAGES_PER_QUERY = 1
S2_FETCH_ABSTRACTS_VIA_BATCH = True
S2_BATCH_SIZE = 200
S2_CACHE_ENABLED = True
S2_CACHE_TTL_DAYS = 30
S2_VERBOSE = False

# -----------------------------
# Retrieval hyperparams (candidate generation)
# -----------------------------
TFIDF_MAX_FEATURES = 200_000
TFIDF_MIN_DF = 2
TFIDF_NGRAM_RANGE = (1, 2)

TOP_PER_QUERY = 250

EMBED_MODEL = "text-embedding-3-small"
MAX_CHARS_PER_EMBED = 3500
EMBED_BATCH_SIZE = 64

W_EMBED = 0.60
W_TFIDF = 0.40
W_EMBED_MAX = 0.70
W_EMBED_BREADTH = 0.30
CITE_WEIGHT = 0.08

# Candidate set construction
CAND_TARGET_N = 220
CAND_TOP_N = 80
CAND_MID_N = 80
CAND_TAIL_N = 60
MID_RANK_RANGE = (81, 500)
TAIL_RANK_RANGE = (501, 2500)
MIN_PER_FACET = 3

# Labeling prompt sizes + concurrency
MAX_ABS_CHARS_FOR_LABEL = 1800
LABEL_CONCURRENCY = 40

print("Config OK")


Config OK


In [4]:
# -----------------------------
# Note
# -----------------------------
# Stage A (OpenAlex + Semantic Scholar fetching) runs after blueprints.
# It writes per-chapter CSVs into `eval_dataset/stageA/`.
print("StageA build happens in the next cells (after blueprints).")


StageA build happens in the next cells (after blueprints).


In [5]:
# -----------------------------
# Chapter specs (3 different chapter types)
# -----------------------------

CHAPTERS: List[Dict[str, Any]] = [
    {
        "chapter_id": "platform_theory",
        "title": "Theoretische Fundierung: Mechanismen der Plattformökonomie",
        "original_text_language": "de",
        "original_text": (
            "Aufarbeitung der Begriffe und Modelle, die zur Analyse von Plattformwachstum und Wettbewerb benötigt werden. "
            "(2.1) Two-/Multi-Sided Markets: Rolle mehrerer Nutzergruppen, Wertschaffung durch Vermittlung. "
            "(2.2) Direkte und indirekte Netzwerkexternalitäten, jeweils positiv und negativ, und ihre Bedeutung für Nutzen, Qualität und Skalierung. "
            "(2.3) Kritische Masse und Wachstumspfad inklusive Henne-Ei-Dilemma. "
            "(2.4) Multi-Homing und Switching Costs als Treiber oder Bremse von Bindung, Marktteilung und Wechselverhalten. "
            "(2.5) Plattform-Governance über Regeln, Rankings und Zugang als Instrument zur Qualitätssteuerung und Stabilisierung der Interaktionen. "
            "(2.6) Monetarisierung und Preisstruktur, einschließlich Gebühren/Provisionen, und deren Wechselwirkung mit Netzwerkeffekten. "
            "(2.7) Wettbewerbsdynamik: Bedingungen, unter denen Netzwerkeffekte „Winner-takes-most“ begünstigen versus Koexistenz mehrerer Plattformen. "
            "Vertiefung erfolgt nur in diesen Konzepten; keine zusätzlichen Markt- oder Organisationstheorien."
        ),
        "scope_hint_en": (
            "Theory foundations of platform economy mechanisms; ONLY the listed concepts; "
            "no additional market/organization theories beyond them."
        ),
        "must_cover_hints": [],
        "must_avoid_hints": ["additional market/organization theories beyond the listed concepts"],
    },
    {
        "chapter_id": "platform_methodology",
        "title": "Methodik: Strukturierte Literaturanalyse und Entwicklung des Frameworks",
        "original_text_language": "de",
        "original_text": (
            "Festlegung eines nachvollziehbaren Vorgehens zur Herleitung und späteren Anwendung eines kompakten Analyse-Frameworks. "
            "(3.1) Strukturierte Literaturanalyse: Definition klarer Suchbegriffe entlang der Themen Two-/Multi-Sided Markets, direkte/indirekte (positive/negative) Netzwerkeffekte, Multi-Homing, Plattform-Governance (Regeln, Rankings, Zugang), Switching Costs, Monetarisierung (Gebühren/Provisionen), Preisstruktur, Wettbewerb und „Winner-takes-most“. "
            "Festlegung transparenter Auswahl- und Ausschlusskriterien mit Fokus auf Passung zu diesen Mechanismen und Verwendbarkeit für ein einheitliches Analysegerüst. "
            "(3.2) Framework-Entwicklung: Verdichtung der Literatur zu den Dimensionen Netzwerkeffekte, Multi-Homing, Governance, Preisstruktur, Wettbewerb. "
            "(3.3) Operationalisierung für die Fallanalyse: Festlegung, wie die Dimensionen mit öffentlich verfügbaren Daten/Dokumenten diskutiert werden (Nutzer-/Anbieterwachstum, Gebührenänderungen, Regeländerungen, Marktanteile, Qualitätsindikatoren). "
            "Auswahl eines konkreten Plattformfalls aus Airbnb, Uber, eBay oder einem App-Store entlang der Verfügbarkeit solcher Informationen."
        ),
        "scope_hint_en": (
            "Method chapter: structured literature review process + framework development + operationalization for case analysis."
        ),
        "must_cover_hints": [],
        "must_avoid_hints": ["unrelated platform theory not used for the framework", "purely technical system architecture"],
    },
    {
        "chapter_id": "platform_empirical_case",
        "title": "Empirische Anwendung: Analyse eines konkreten Plattformfalls mit dem Framework",
        "original_text_language": "de",
        "original_text": (
            "Darstellung und Analyse des ausgewählten Plattformfalls (Airbnb, Uber, eBay oder ein App-Store) anhand öffentlich verfügbarer Daten und Dokumente. "
            "(4.1) Kurzprofil der Plattform und der relevanten Nutzergruppen sowie Abgrenzung des betrachteten Marktbereichs, soweit für die Framework-Anwendung notwendig. "
            "(4.2) Anwendung des Frameworks entlang der Dimensionen: Identifikation und Einordnung direkter und indirekter Netzwerkeffekte (positiv/negativ) und deren Zusammenhang mit Nutzer-/Anbieterwachstum und dem Erreichen bzw. Überschreiten kritischer Masse; Beobachtung von Multi-Homing und möglichen Wechselbarrieren (Switching Costs); "
            "Analyse von Governance-Änderungen (Regeln, Rankings, Zugang) und deren Bezug zu Qualitätsindikatoren; "
            "Einordnung der Preisstruktur und Monetarisierung über Gebühren/Provisionen inklusive relevanter Gebührenänderungen; "
            "Diskussion der Wettbewerbsentwicklung anhand verfügbarer Marktanteilsindikatoren. "
            "(4.3) Fallbezogene Synthese, welche Netzwerkeffekte im Fall dominieren und welche Konsequenzen sich für Strategie und ggf. Regulierung ableiten lassen. "
            "Keine neue Theorieentwicklung über das Framework hinaus."
        ),
        "scope_hint_en": (
            "Empirical application: apply the framework to a specific platform using public data/documents; no new theory development."
        ),
        "must_cover_hints": [],
        "must_avoid_hints": ["introducing new theory beyond the framework"],
    },
]

print("Chapters:")
for c in CHAPTERS:
    print("-", c["chapter_id"], "|", c["title"])


Chapters:
- platform_theory | Theoretische Fundierung: Mechanismen der Plattformökonomie
- platform_methodology | Methodik: Strukturierte Literaturanalyse und Entwicklung des Frameworks
- platform_empirical_case | Empirische Anwendung: Analyse eines konkreten Plattformfalls mit dem Framework


In [6]:
# -----------------------------
# Blueprint generation (cached to disk)
# -----------------------------

class ChapterBlueprint(BaseModel):
    chapter_id: str
    language: str = Field("en")

    scope_statement: str
    must_cover: List[str]
    should_cover: List[str]
    must_avoid: List[str]

    main_query: str
    facet_queries: List[str]
    keywords: List[str]
    key_concepts: List[str]

    preferred_source_types: Optional[List[str]] = None
    negative_query_terms: Optional[List[str]] = None

    scoring_guidance: str
    notes: Optional[str] = None

def price_for_model(model: str) -> dict:
    return MODEL_PRICES_USD_PER_1M.get(model, {"input": 0.0, "cached": 0.0, "output": 0.0})

def cost_from_usage(usage, model: str) -> dict:
    """Token totals + estimated cost using MODEL_PRICES_USD_PER_1M."""

    prices = price_for_model(model)

    def req_cost(input_tokens, cached_tokens, output_tokens):
        cached_tokens = int(cached_tokens or 0)
        input_tokens = int(input_tokens or 0)
        output_tokens = int(output_tokens or 0)

        non_cached = max(0, input_tokens - cached_tokens)
        cost = (
            (non_cached / 1_000_000) * prices["input"]
            + (cached_tokens / 1_000_000) * prices["cached"]
            + (output_tokens / 1_000_000) * prices["output"]
        )
        return non_cached, cached_tokens, output_tokens, cost

    entries = getattr(usage, "request_usage_entries", None) or []
    if entries:
        total_in = total_cached = total_out = 0
        total_cost = 0.0
        for r in entries:
            inp = getattr(r, "input_tokens", 0) or 0
            out = getattr(r, "output_tokens", 0) or 0
            itd = getattr(r, "input_tokens_details", None)
            cached = getattr(itd, "cached_tokens", 0) if itd is not None else 0
            non_cached, cached, out, c = req_cost(inp, cached, out)
            total_in += non_cached
            total_cached += cached
            total_out += out
            total_cost += c
        return {
            "requests": int(getattr(usage, "requests", len(entries)) or len(entries)),
            "input_tokens": int(total_in + total_cached),
            "cached_input_tokens": int(total_cached),
            "output_tokens": int(total_out),
            "cost_usd": float(total_cost),
        }

    # Fallback: aggregated totals
    inp = int(getattr(usage, "input_tokens", 0) or 0)
    out = int(getattr(usage, "output_tokens", 0) or 0)
    itd = getattr(usage, "input_tokens_details", None)
    cached = int(getattr(itd, "cached_tokens", 0) if itd is not None else 0)

    non_cached, cached, out, c = req_cost(inp, cached, out)
    return {
        "requests": int(getattr(usage, "requests", 1) or 1),
        "input_tokens": int(non_cached + cached),
        "cached_input_tokens": int(cached),
        "output_tokens": int(out),
        "cost_usd": float(c),
    }

blueprint_agent = Agent(
    name="Chapter Blueprint Builder",
    model=BLUEPRINT_MODEL,
    model_settings=ModelSettings(top_p=1.0, verbosity="low"),
    instructions=(
        "You create a chapter blueprint (rubric) and search queries for academic literature retrieval.\n"
        "Return ONLY the structured output fields (no extra text).\n\n"
        "Constraints:\n"
        "- language must be 'en'\n"
        "- scope_statement: 1 sentence, <= 30 words\n"
        "- must_cover: 4–8 bullets, each <= 16 words\n"
        "- should_cover: 3–8 bullets, each <= 16 words\n"
        "- must_avoid: 3–8 bullets, each <= 16 words\n"
        "- main_query: <= 18 words\n"
        "- facet_queries: 8–14 items, each <= 14 words\n"
        "- keywords: 20–45 items\n"
        "- key_concepts: 10–22 items\n"
        "- preferred_source_types: 2–6 items\n"
        "- negative_query_terms: 0–12 items derived from must_avoid (soft negatives)\n"
        "- scoring_guidance: <= 80 words\n"
        "- Do NOT contradict yourself: if something is in must_avoid, do not emphasize it in keywords/facets.\n"
        "- Do NOT hardcode any domain. Follow the given chapter spec.\n"
    ),
    output_type=ChapterBlueprint,
)

async def get_or_create_blueprint(chapter: dict) -> tuple[dict, dict]:
    out_path = BLUEPRINT_DIR / f"{chapter['chapter_id']}.json"
    if out_path.exists():
        bp = json.loads(out_path.read_text(encoding="utf-8"))
        return bp, {"requests": 0, "input_tokens": 0, "cached_input_tokens": 0, "output_tokens": 0, "cost_usd": 0.0}

    prompt = (
        "Create a ChapterBlueprint for academic literature retrieval.\n"
        "Return ONLY the structured output fields required by the schema.\n\n"
        "CHAPTER_SPEC_JSON:\n"
        + json.dumps(chapter, ensure_ascii=False, indent=2)
    )

    res = await Runner.run(blueprint_agent, prompt)
    bp = res.final_output.model_dump()
    out_path.write_text(json.dumps(bp, ensure_ascii=False, indent=2), encoding="utf-8")

    usage = res.context_wrapper.usage
    return bp, cost_from_usage(usage, model=BLUEPRINT_MODEL)

blueprints: Dict[str, dict] = {}
bp_totals = {"requests": 0, "input_tokens": 0, "cached_input_tokens": 0, "output_tokens": 0, "cost_usd": 0.0}
for ch in CHAPTERS:
    bp, u = await get_or_create_blueprint(ch)
    blueprints[ch["chapter_id"]] = bp
    for k in bp_totals:
        bp_totals[k] += u.get(k, 0)

print("Blueprints ready")
print("Blueprint cost summary:", bp_totals)
for cid, bp in blueprints.items():
    print("-", cid, "| facets:", len(bp.get("facet_queries", [])), "| main:", bp.get("main_query"))


Blueprints ready
Blueprint cost summary: {'requests': 3, 'input_tokens': 2687, 'cached_input_tokens': 0, 'output_tokens': 7646, 'cost_usd': 0.01596375}
- platform_theory | facets: 12 | main: platform network effects multi-sided markets governance monetization multi-homing competition
- platform_methodology | facets: 12 | main: platform markets literature review network effects multi-homing governance pricing competition
- platform_empirical_case | facets: 10 | main: platform empirical case study network effects governance pricing market share public data


In [7]:
# -----------------------------
# Stage A: fetch OpenAlex + Semantic Scholar per chapter
# and build per-chapter StageA CSVs
# -----------------------------

# Output files:
# - eval_dataset/fetch/openalex_<chapter_id>.csv
# - eval_dataset/fetch/semantic_scholar_<chapter_id>.csv
# - eval_dataset/stageA/stageA_combined_oa_s2_<chapter_id>.csv
# - eval_dataset/stageA/stageA_all_chapters.csv

def dedupe_preserve_order(items: List[str]) -> List[str]:
    seen = set()
    out = []
    for x in items:
        x = str(x).strip()
        if not x:
            continue
        k = x.lower()
        if k in seen:
            continue
        seen.add(k)
        out.append(x)
    return out

def query_list_for_chapter(bp: dict) -> List[str]:
    qs = [bp.get("main_query", "")] + list(bp.get("facet_queries") or [])
    qs = dedupe_preserve_order(qs)
    return qs[:FETCH_MAX_QUERIES_PER_CHAPTER]

# ----------------------------
# OpenAlex
# ----------------------------

def abstract_from_inverted_index(inv: Optional[Dict[str, List[int]]]) -> Optional[str]:
    if not inv:
        return None

    pairs: List[tuple[int, str]] = []
    for word, positions in inv.items():
        if not positions:
            continue
        for p in positions:
            if isinstance(p, int):
                pairs.append((p, word))

    if not pairs:
        return None

    pairs.sort(key=lambda x: x[0])
    max_pos = pairs[-1][0]
    words = [""] * (max_pos + 1)
    for pos, w in pairs:
        if 0 <= pos <= max_pos:
            words[pos] = w

    text = " ".join(w for w in words if w).strip()
    return text or None

def venue_from_primary_location(work: Dict[str, Any]) -> Optional[str]:
    pl = work.get("primary_location") or {}
    src = pl.get("source") or {}
    return src.get("display_name")

def first_n_authors(work: Dict[str, Any], n: int = 6) -> str:
    authors = []
    for a in (work.get("authorships") or [])[:n]:
        name = ((a.get("author") or {}).get("display_name"))
        if name:
            authors.append(name)
    return "; ".join(authors)

def oa_get(params: Dict[str, Any], max_retries: int = 6) -> dict:
    backoff = 1.0
    for attempt in range(1, max_retries + 1):
        r = requests.get(OA_BASE_URL, params=params, timeout=OA_TIMEOUT_SEC)
        if r.status_code in (429, 500, 502, 503, 504):
            if attempt == max_retries:
                raise RuntimeError(f"OpenAlex error {r.status_code} | URL: {r.url} | Body: {r.text[:400]}")
            time.sleep(backoff)
            backoff *= 2
            continue
        if r.status_code >= 400:
            raise RuntimeError(f"OpenAlex error {r.status_code} | URL: {r.url} | Body: {r.text[:400]}")
        return r.json()
    raise RuntimeError("OpenAlex retry loop exhausted")

def fetch_openalex_query(q: str, max_works: Optional[int]) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    cursor = "*"
    select = (
        "id,display_name,publication_year,type,doi,cited_by_count,"
        "authorships,primary_location,abstract_inverted_index"
    )

    while cursor:
        params: Dict[str, Any] = {
            "search": q,
            "per-page": OA_PER_PAGE,
            "cursor": cursor,
            "select": select,
        }
        if OPENALEX_API_KEY:
            params["api_key"] = OPENALEX_API_KEY

        data = oa_get(params)

        for w in data.get("results", []) or []:
            rows.append({
                "query": q,
                "title": w.get("display_name"),
                "year": w.get("publication_year"),
                "type": w.get("type"),
                "venue": venue_from_primary_location(w),
                "cited_by": w.get("cited_by_count"),
                "authors(first6)": first_n_authors(w, n=6),
                "doi": w.get("doi"),
                "openalex_id": w.get("id"),
                "abstract": abstract_from_inverted_index(w.get("abstract_inverted_index")),
            })

            if max_works is not None and len(rows) >= max_works:
                return rows

        cursor = (data.get("meta") or {}).get("next_cursor")
        if not cursor:
            break

    return rows

def fetch_openalex_for_chapter(chapter_id: str, queries: List[str]) -> pd.DataFrame:
    all_rows: List[Dict[str, Any]] = []
    for qi, q in enumerate(queries, start=1):
        print(f"[OpenAlex:{chapter_id}] ({qi}/{len(queries)}) {q}")
        all_rows.extend(fetch_openalex_query(q, max_works=OA_MAX_WORKS_PER_QUERY))

    df_oa = pd.DataFrame(all_rows)
    if df_oa.empty:
        return df_oa

    df_oa.insert(0, "chapter_id", chapter_id)
    df_oa = df_oa.drop_duplicates(subset=["chapter_id", "query", "openalex_id"], keep="first").reset_index(drop=True)
    return df_oa

# ----------------------------
# Semantic Scholar
# ----------------------------

S2_CACHE_DIR = FETCH_DIR / "s2_cache"
S2_CACHE_DIR.mkdir(exist_ok=True)

def _now_utc() -> datetime:
    return datetime.now(timezone.utc)

def _cache_key(method: str, url: str, params: Optional[Dict[str, Any]], body: Optional[Dict[str, Any]]) -> str:
    blob = {"m": method.upper(), "u": url, "p": params or {}, "b": body or {}}
    s = json.dumps(blob, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha1(s).hexdigest()

def s2_cache_get(method: str, url: str, params: Optional[Dict[str, Any]], body: Optional[Dict[str, Any]]) -> Optional[Any]:
    if not S2_CACHE_ENABLED:
        return None
    key = _cache_key(method, url, params, body)
    path = S2_CACHE_DIR / f"{key}.json"
    if not path.exists():
        return None
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
        created = datetime.fromisoformat(payload["created"])
        if _now_utc() - created > timedelta(days=S2_CACHE_TTL_DAYS):
            return None
        return payload["data"]
    except Exception:
        return None

def s2_cache_set(method: str, url: str, params: Optional[Dict[str, Any]], body: Optional[Dict[str, Any]], data: Any) -> None:
    if not S2_CACHE_ENABLED:
        return
    key = _cache_key(method, url, params, body)
    path = S2_CACHE_DIR / f"{key}.json"
    payload = {"created": _now_utc().isoformat(), "data": data}
    path.write_text(json.dumps(payload), encoding="utf-8")

def _parse_retry_after(resp: requests.Response) -> Optional[float]:
    ra = resp.headers.get("Retry-After")
    if not ra:
        return None
    try:
        return float(ra)
    except ValueError:
        return None

def s2_request(session: requests.Session, method: str, url: str, params: Optional[Dict[str, Any]] = None, body: Optional[Dict[str, Any]] = None, max_retries: int = 8) -> Any:
    cached = s2_cache_get(method, url, params, body)
    if cached is not None:
        return cached

    backoff = 1.0
    for attempt in range(1, max_retries + 1):
        resp = session.request(method, url, params=params, json=body, timeout=30)
        if resp.status_code == 200:
            data = resp.json()
            s2_cache_set(method, url, params, body, data)
            return data

        if resp.status_code in (429, 500, 502, 503, 504):
            ra = _parse_retry_after(resp)
            wait = max(ra or 0.0, backoff)
            if S2_VERBOSE:
                print(f"[S2] {resp.status_code} retry in {wait:.1f}s | attempt {attempt}/{max_retries}")
            time.sleep(wait)
            backoff = min(30.0, backoff * 2)
            continue

        raise RuntimeError(f"S2 error {resp.status_code} | body: {resp.text[:500]}")

    raise RuntimeError(f"S2 retry loop exhausted: {method} {url}")

def clean_query(q: str) -> str:
    q = q.replace("-", " ")
    q = re.sub(r"\s+", " ", q).strip()
    return q

def chunks(xs: List[str], n: int) -> Iterable[List[str]]:
    for i in range(0, len(xs), n):
        yield xs[i:i+n]

def fetch_s2_for_chapter(chapter_id: str, queries: List[str]) -> pd.DataFrame:
    SEARCH_FIELDS = "paperId,title,year,authors,venue,citationCount,externalIds,url"
    DETAIL_FIELDS = "paperId,abstract"

    session = requests.Session()
    session.headers.update({"User-Agent": "instantpaper-eval/1.0"})
    if S2_API_KEY:
        session.headers.update({"x-api-key": S2_API_KEY})

    rows: List[Dict[str, Any]] = []

    for qi, q in enumerate(queries, start=1):
        q2 = clean_query(q)
        print(f"[S2:{chapter_id}] ({qi}/{len(queries)}) {q2}")

        offset = 0
        for _page in range(1, S2_MAX_PAGES_PER_QUERY + 1):
            params = {
                "query": q2,
                "fields": SEARCH_FIELDS,
                "limit": S2_LIMIT,
                "offset": offset,
            }
            data = s2_request(session, "GET", S2_SEARCH_URL, params=params, body=None, max_retries=10)
            papers = data.get("data", []) or []

            for p in papers:
                authors = [a.get("name") for a in (p.get("authors") or [])[:6] if a.get("name")]
                ext = p.get("externalIds") or {}
                rows.append({
                    "chapter_id": chapter_id,
                    "query": q,
                    "paperId": p.get("paperId"),
                    "title": p.get("title"),
                    "year": p.get("year"),
                    "venue": p.get("venue"),
                    "citationCount": p.get("citationCount"),
                    "authors(first6)": "; ".join(authors),
                    "doi": ext.get("DOI"),
                    "s2_url": p.get("url"),
                })

            if len(papers) < S2_LIMIT:
                break
            offset += S2_LIMIT

    df_s2 = pd.DataFrame(rows)
    if df_s2.empty:
        return df_s2

    df_s2 = df_s2.drop_duplicates(subset=["chapter_id", "query", "paperId"]).reset_index(drop=True)

    if S2_FETCH_ABSTRACTS_VIA_BATCH:
        unique_ids = [pid for pid in df_s2["paperId"].dropna().unique().tolist() if pid]
        abstracts: Dict[str, Optional[str]] = {}

        for ids in chunks(unique_ids, S2_BATCH_SIZE):
            params = {"fields": DETAIL_FIELDS}
            body = {"ids": ids}
            batch = s2_request(session, "POST", S2_BATCH_URL, params=params, body=body, max_retries=10)

            if isinstance(batch, list):
                for p in batch:
                    pid = p.get("paperId")
                    if pid:
                        abstracts[pid] = p.get("abstract")
            else:
                for p in (batch.get("data", []) or []):
                    pid = p.get("paperId")
                    if pid:
                        abstracts[pid] = p.get("abstract")

        df_s2["abstract"] = df_s2["paperId"].map(abstracts)

    return df_s2

# ----------------------------
# Stage A merge (OpenAlex + S2) per chapter
# ----------------------------

def standardize_openalex(df_in: pd.DataFrame) -> pd.DataFrame:
    d = df_in.copy()
    d["source"] = "openalex"
    d["source_id"] = d.get("openalex_id")
    d["citation_count"] = d.get("cited_by")
    for col in ["abstract", "doi", "title", "year", "venue", "type", "authors(first6)", "query", "chapter_id", "openalex_id"]:
        if col not in d.columns:
            d[col] = np.nan
    return d[[
        "chapter_id",
        "source", "source_id", "query", "title", "year", "venue", "type",
        "authors(first6)", "doi", "citation_count", "abstract", "openalex_id"
    ]]

def standardize_s2(df_in: pd.DataFrame) -> pd.DataFrame:
    d = df_in.copy()
    d["source"] = "semantic_scholar"
    d["source_id"] = d.get("paperId")
    d["citation_count"] = d.get("citationCount")
    for col in ["abstract", "doi", "title", "year", "venue", "authors(first6)", "query", "paperId", "s2_url", "chapter_id"]:
        if col not in d.columns:
            d[col] = np.nan
    d["type"] = np.nan
    d["openalex_id"] = np.nan
    return d[[
        "chapter_id",
        "source", "source_id", "query", "title", "year", "venue", "type",
        "authors(first6)", "doi", "citation_count", "abstract", "paperId", "s2_url"
    ]]

def normalize_doi(x):
    if not isinstance(x, str):
        return None
    x = x.strip().lower()
    if not x:
        return None
    x = re.sub(r"^https?://(dx\\.)?doi\\.org/", "", x)
    x = re.sub(r"^doi:\\s*", "", x)
    x = x.strip()
    return x or None

def normalize_title(x):
    if not isinstance(x, str):
        return None
    x = x.strip().lower()
    if not x:
        return None
    x = re.sub(r"[\\u2010-\\u2015]", "-", x)
    x = re.sub(r"[^a-z0-9\\s]", " ", x)
    x = re.sub(r"\\s+", " ", x).strip()
    return x or None

def longest_text(series):
    vals = [v for v in series if isinstance(v, str) and v.strip()]
    if not vals:
        return None
    return max(vals, key=len)

def most_common_or_longest(series):
    vals = [v for v in series if isinstance(v, str) and v.strip()]
    if not vals:
        return None
    vc = pd.Series(vals).value_counts()
    if len(vc) and vc.iloc[0] >= 2:
        return vc.index[0]
    return max(vals, key=len)

def first_nonnull(series):
    for v in series:
        if pd.notna(v) and v not in ("", None):
            return v
    return None

def within_source_key(df_in: pd.DataFrame) -> pd.Series:
    k = []
    for _, r in df_in.iterrows():
        if r.get("doi_norm"):
            k.append(f"doi:{r['doi_norm']}")
        elif pd.notna(r.get("source_id")):
            k.append(f"id:{r['source']}:{r['source_id']}")
        else:
            y = int(r["year"]) if pd.notna(r.get("year")) else ""
            k.append(f"ty:{r['title_norm']}|{y}")
    return pd.Series(k, index=df_in.index)

def cross_source_merge_key(r: pd.Series) -> str:
    if isinstance(r.get("doi_norm"), str) and r.get("doi_norm"):
        return f"doi:{r['doi_norm']}"
    if pd.notna(r.get("year")):
        return f"ty:{r['title_norm']}|{int(round(float(r['year'])))}"
    return f"t:{r['title_norm']}"

def build_stagea(df_oa_raw: pd.DataFrame, df_s2_raw: pd.DataFrame, chapter_id: str) -> pd.DataFrame:
    oa_std = standardize_openalex(df_oa_raw)
    s2_std = standardize_s2(df_s2_raw)

    combined_raw = pd.concat([oa_std, s2_std], ignore_index=True)
    stageA = combined_raw.copy()

    stageA["doi_norm"] = stageA["doi"].map(normalize_doi)
    stageA["title_norm"] = stageA["title"].map(normalize_title)
    stageA["citation_count"] = pd.to_numeric(stageA["citation_count"], errors="coerce")

    stageA = stageA[stageA["title_norm"].notna()].reset_index(drop=True)
    stageA["within_key"] = within_source_key(stageA)

    agg_map = {
        "chapter_id": lambda s: s.iloc[0],
        "source": lambda s: s.iloc[0],
        "source_id": first_nonnull,
        "title": most_common_or_longest,
        "title_norm": lambda s: s.iloc[0],
        "year": lambda s: pd.to_numeric(s, errors="coerce").dropna().median() if s.notna().any() else np.nan,
        "venue": most_common_or_longest,
        "type": most_common_or_longest,
        "authors(first6)": most_common_or_longest,
        "doi": first_nonnull,
        "doi_norm": first_nonnull,
        "citation_count": lambda s: pd.to_numeric(s, errors="coerce").max(),
        "abstract": longest_text,
        "query": lambda s: "; ".join(sorted(set([q for q in s if isinstance(q, str) and q.strip()]))),
        "openalex_id": first_nonnull,
        "paperId": first_nonnull,
        "s2_url": first_nonnull,
    }

    stageA_dedup = (
        stageA
        .groupby(["source", "within_key"], as_index=False)
        .agg(agg_map)
        .drop(columns=["within_key"])
    )

    stageA_dedup["merge_key"] = stageA_dedup.apply(cross_source_merge_key, axis=1)

    def merge_sources(group: pd.DataFrame) -> dict:
        sources = sorted(set(group["source"].dropna().tolist()))
        source_ids = {src: group.loc[group["source"] == src, "source_id"].dropna().astype(str).unique().tolist() for src in sources}

        return {
            "chapter_id": chapter_id,
            "merge_key": group["merge_key"].iloc[0],
            "sources": "; ".join(sources),
            "source_count": len(sources),
            "source_ids": str(source_ids),
            "title": most_common_or_longest(group["title"]),
            "year": pd.to_numeric(group["year"], errors="coerce").dropna().median() if group["year"].notna().any() else np.nan,
            "venue": most_common_or_longest(group["venue"]),
            "type": most_common_or_longest(group["type"]),
            "authors(first6)": most_common_or_longest(group["authors(first6)"]),
            "doi": first_nonnull(group["doi"]),
            "doi_norm": first_nonnull(group["doi_norm"]),
            "citation_count_max": pd.to_numeric(group["citation_count"], errors="coerce").max(),
            "abstract": longest_text(group["abstract"]),
            "queries": "; ".join(sorted(set(
                q for q in group["query"].dropna().tolist()
                if isinstance(q, str) and q.strip()
            ))),
            "openalex_id": first_nonnull(group.get("openalex_id", pd.Series([], dtype=object))),
            "paperId": first_nonnull(group.get("paperId", pd.Series([], dtype=object))),
            "s2_url": first_nonnull(group.get("s2_url", pd.Series([], dtype=object))),
        }

    merged_records = [merge_sources(g) for _, g in stageA_dedup.groupby("merge_key")]
    df_stageA = pd.DataFrame(merged_records)

    df_stageA["merge_kind"] = df_stageA["merge_key"].str.split(":", n=1).str[0]
    df_stageA["has_abstract"] = df_stageA["abstract"].notna() & (df_stageA["abstract"].astype(str).str.len() > 50)
    df_stageA = df_stageA.sort_values(by="citation_count_max", ascending=False, na_position="last").reset_index(drop=True)

    return df_stageA

# ----------------------------
# Run for each chapter
# ----------------------------

stageA_frames = []

for ch in CHAPTERS:
    cid = ch["chapter_id"]
    bp = blueprints[cid]
    queries = query_list_for_chapter(bp)

    print(f"\n=== Stage A for chapter: {cid} | queries={len(queries)} ===")

    oa_csv = FETCH_DIR / f"openalex_{cid}.csv"
    s2_csv = FETCH_DIR / f"semantic_scholar_{cid}.csv"
    stagea_csv = STAGEA_DIR / f"stageA_combined_oa_s2_{cid}.csv"

    # OpenAlex
    if RUN_OPENALEX:
        if oa_csv.exists() and not FETCH_FORCE:
            df_oa = pd.read_csv(oa_csv)
            print(f"[OpenAlex:{cid}] cached rows: {len(df_oa)}")
        else:
            df_oa = fetch_openalex_for_chapter(cid, queries)
            df_oa.to_csv(oa_csv, index=False, encoding="utf-8")
            print(f"[OpenAlex:{cid}] saved: {oa_csv} | rows={len(df_oa)}")
    else:
        df_oa = pd.DataFrame(columns=["chapter_id", "query", "title", "year", "type", "venue", "cited_by", "authors(first6)", "doi", "openalex_id", "abstract"])

    # Semantic Scholar
    if RUN_SEMANTIC_SCHOLAR:
        if s2_csv.exists() and not FETCH_FORCE:
            df_s2 = pd.read_csv(s2_csv)
            print(f"[S2:{cid}] cached rows: {len(df_s2)}")
        else:
            df_s2 = fetch_s2_for_chapter(cid, queries)
            df_s2.to_csv(s2_csv, index=False, encoding="utf-8")
            print(f"[S2:{cid}] saved: {s2_csv} | rows={len(df_s2)}")
    else:
        df_s2 = pd.DataFrame(columns=["chapter_id", "query", "paperId", "title", "year", "venue", "citationCount", "authors(first6)", "doi", "s2_url", "abstract"])

    # StageA merge
    if stagea_csv.exists() and not FETCH_FORCE:
        df_stageA = pd.read_csv(stagea_csv)
        print(f"[StageA:{cid}] cached rows: {len(df_stageA)}")
    else:
        df_stageA = build_stagea(df_oa, df_s2, chapter_id=cid)
        df_stageA.to_csv(stagea_csv, index=False, encoding="utf-8")
        print(f"[StageA:{cid}] saved: {stagea_csv} | rows={len(df_stageA)}")

    stageA_frames.append(df_stageA)

# Save union (for convenience)
df_stageA_all = pd.concat(stageA_frames, axis=0).reset_index(drop=True)
df_stageA_all.to_csv(STAGEA_ALL_CSV, index=False, encoding="utf-8")
print(f"\nSaved all-chapters StageA: {STAGEA_ALL_CSV} | rows={len(df_stageA_all)}")

df_stageA_all.groupby(["chapter_id", "sources"]).size().sort_values(ascending=False).head(15)



=== Stage A for chapter: platform_theory | queries=13 ===
[OpenAlex:platform_theory] (1/13) platform network effects multi-sided markets governance monetization multi-homing competition
[OpenAlex:platform_theory] (2/13) two-sided markets model literature review
[OpenAlex:platform_theory] (3/13) multi-sided platforms value creation mechanisms
[OpenAlex:platform_theory] (4/13) direct and indirect network externalities positive negative
[OpenAlex:platform_theory] (5/13) critical mass tipping points chicken-egg dilemma
[OpenAlex:platform_theory] (6/13) multi-homing switching costs market share effects
[OpenAlex:platform_theory] (7/13) platform governance rules rankings access control
[OpenAlex:platform_theory] (8/13) monetization pricing fees commissions network effects
[OpenAlex:platform_theory] (9/13) winner-takes-most dynamics platform coexistence conditions
[OpenAlex:platform_theory] (10/13) empirical identification of network externalities
[OpenAlex:platform_theory] (11/13) formal gr

chapter_id               sources                   
platform_theory          openalex                      3247
platform_methodology     openalex                      3195
platform_empirical_case  openalex                      2699
platform_theory          semantic_scholar              1175
platform_methodology     semantic_scholar              1099
platform_empirical_case  semantic_scholar              1080
platform_theory          openalex; semantic_scholar       8
platform_empirical_case  openalex; semantic_scholar       2
platform_methodology     openalex; semantic_scholar       1
dtype: int64

In [13]:
# -----------------------------
# Candidate pool utilities
# -----------------------------

def minmax(x):
    x = np.asarray(x, dtype=float)
    mn, mx = np.nanmin(x), np.nanmax(x)
    return (x - mn) / (mx - mn + 1e-12)

def truncate_text(s: str, max_chars: int) -> str:
    s = "" if s is None else str(s)
    s = s.replace("\n", " ").strip()
    return s[:max_chars]

def build_chapter_query_text(bp: dict) -> str:
    parts = []
    main = bp["main_query"]
    parts.extend([main, main])
    parts.extend(bp.get("facet_queries", []))
    parts.extend(bp.get("keywords", []))
    parts.extend(bp.get("key_concepts", []))
    return " ".join(parts)

def tfidf_scores(query_text: str, vectorizer: TfidfVectorizer, X) -> np.ndarray:
    qv = vectorizer.transform([query_text])
    return cosine_similarity(qv, X).ravel()

def facet_union_pool(bp: dict, df_in: pd.DataFrame, vectorizer: TfidfVectorizer, X, top_per_query: int) -> pd.DataFrame:
    query_texts = [bp["main_query"], bp["main_query"]] + list(bp.get("facet_queries", []))
    pool_keys = set()
    for qt in query_texts:
        qv = vectorizer.transform([qt])
        sims = cosine_similarity(qv, X).ravel()
        top_idx = np.argsort(-sims)[:top_per_query]
        pool_keys.update(df_in.iloc[top_idx]["merge_key"].astype(str).tolist())
    out = df_in[df_in["merge_key"].astype(str).isin(pool_keys)].copy()
    out = out.drop_duplicates(subset=["merge_key"]).reset_index(drop=True)
    return out

def l2_normalize(mat: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(mat, axis=1, keepdims=True) + 1e-12
    return mat / norms

def hash_obj(obj: Any) -> str:
    blob = json.dumps(obj, ensure_ascii=False, sort_keys=True)
    return hashlib.sha1(blob.encode("utf-8")).hexdigest()

def save_npz(path: Path, keys: List[str], mat: np.ndarray):
    np.savez_compressed(path, keys=np.array(keys, dtype=object), embeds=mat.astype(np.float32))

def load_npz(path: Path):
    z = np.load(path, allow_pickle=True)
    return list(z["keys"]), z["embeds"].astype(np.float32)

embed_totals = {"requests": 0, "input_tokens": 0, "cost_usd": 0.0}

def embed_texts(texts: List[str], model: str, batch_size: int, max_retries: int = 6) -> np.ndarray:
    """Embeddings with retries + token/cost tracking."""
    price = MODEL_PRICES_USD_PER_1M.get(model, {"input": 0.0}).get("input", 0.0)
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        backoff = 1.0
        for attempt in range(1, max_retries + 1):
            try:
                resp = client.embeddings.create(model=model, input=batch)
                vecs = [np.array(item.embedding, dtype=np.float32) for item in resp.data]
                all_vecs.extend(vecs)

                usage = getattr(resp, "usage", None)
                tok = int(getattr(usage, "total_tokens", 0) or getattr(usage, "prompt_tokens", 0) or 0)
                embed_totals["requests"] += 1
                embed_totals["input_tokens"] += tok
                embed_totals["cost_usd"] += (tok / 1_000_000) * price
                break
            except Exception:
                if attempt == max_retries:
                    raise
                time.sleep(backoff)
                backoff *= 2
    return np.vstack(all_vecs)

def score_pool_with_embeddings(bp: dict, pool: pd.DataFrame) -> pd.DataFrame:
    pool = pool.copy()
    pool["doc_text_trunc"] = (
        pool["title"].fillna("") + "\n\n" + pool["abstract"].fillna("")
    ).apply(lambda s: truncate_text(s, MAX_CHARS_PER_EMBED))

    keys = pool["merge_key"].astype(str).tolist()

    # Doc embeddings cache
    doc_hash = hash_obj({"model": EMBED_MODEL, "max_chars": MAX_CHARS_PER_EMBED, "keys": keys})[:16]
    doc_npz = EMBED_CACHE_DIR / f"eval_doc_embeds_{EMBED_MODEL}_{doc_hash}.npz"

    # Query embeddings cache
    query_texts = [bp["main_query"], bp["main_query"]] + list(bp.get("facet_queries", []))
    q_hash = hash_obj({"model": EMBED_MODEL, "queries": query_texts})[:16]
    q_json = EMBED_CACHE_DIR / f"eval_query_embeds_{EMBED_MODEL}_{q_hash}.json"

    # Docs
    if doc_npz.exists():
        cached_keys, doc_embeds = load_npz(doc_npz)
        if cached_keys != keys:
            doc_embeds = embed_texts(pool["doc_text_trunc"].tolist(), model=EMBED_MODEL, batch_size=EMBED_BATCH_SIZE)
            save_npz(doc_npz, keys, doc_embeds)
    else:
        doc_embeds = embed_texts(pool["doc_text_trunc"].tolist(), model=EMBED_MODEL, batch_size=EMBED_BATCH_SIZE)
        save_npz(doc_npz, keys, doc_embeds)

    # Queries
    if q_json.exists():
        q_cached = json.loads(q_json.read_text(encoding="utf-8"))
        query_embeds = np.array(q_cached["embeddings"], dtype=np.float32)
    else:
        query_embeds = embed_texts(query_texts, model=EMBED_MODEL, batch_size=EMBED_BATCH_SIZE)
        q_json.write_text(
            json.dumps({"model": EMBED_MODEL, "query_texts": query_texts, "embeddings": query_embeds.tolist()}, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

    docN = l2_normalize(doc_embeds)
    qN = l2_normalize(query_embeds)
    S = docN @ qN.T

    pool["score_embed_max"] = S.max(axis=1)
    pool["score_embed_mean_top3"] = np.sort(S, axis=1)[:, -3:].mean(axis=1)
    pool["score_embed_norm"] = minmax(pool["score_embed_max"].values)
    pool["score_embed_mean_top3_norm"] = minmax(pool["score_embed_mean_top3"].values)
    pool["score_embed_combo"] = W_EMBED_MAX * pool["score_embed_norm"] + W_EMBED_BREADTH * pool["score_embed_mean_top3_norm"]

    # Facet assignment (exclude the two main-query columns)
    facets = list(bp.get("facet_queries", []))
    if facets:
        facet_S = S[:, 2:2+len(facets)]
        facet_best_i = facet_S.argmax(axis=1).astype(int)
        pool["facet_best_i"] = facet_best_i
        pool["facet_best_query"] = [facets[i] for i in facet_best_i]
    else:
        pool["facet_best_i"] = -1
        pool["facet_best_query"] = ""

    return pool

def build_candidate_set(pool_scored: pd.DataFrame, rank_col: str, bp: dict) -> pd.DataFrame:
    pool_scored = pool_scored.sort_values(rank_col, ascending=False).reset_index(drop=True).copy()
    pool_scored["rank"] = np.arange(1, len(pool_scored) + 1)

    top = pool_scored.head(min(CAND_TOP_N, len(pool_scored)))

    mid_start, mid_end = MID_RANK_RANGE
    mid = pool_scored[(pool_scored["rank"] >= mid_start) & (pool_scored["rank"] <= min(mid_end, len(pool_scored)))]

    tail_start, tail_end = TAIL_RANK_RANGE
    tail = pool_scored[(pool_scored["rank"] >= tail_start) & (pool_scored["rank"] <= min(tail_end, len(pool_scored)))]

    mid_n = min(CAND_MID_N, len(mid))
    tail_n = min(CAND_TAIL_N, len(tail))
    mid_s = mid.sample(n=mid_n, random_state=SEED) if mid_n > 0 else mid.head(0)
    tail_s = tail.sample(n=tail_n, random_state=SEED) if tail_n > 0 else tail.head(0)

    picked = pd.concat([top, mid_s, tail_s], axis=0).drop_duplicates(subset=["merge_key"]).reset_index(drop=True)

    # Facet coverage
    facets = list(bp.get("facet_queries", []))
    if facets and "facet_best_i" in pool_scored.columns:
        def weakest_replaceable(df_pick: pd.DataFrame) -> Optional[str]:
            repl = df_pick[df_pick["rank"] > CAND_TOP_N].sort_values(rank_col, ascending=True)
            if repl.empty:
                return None
            return str(repl.iloc[0]["merge_key"])

        for i in range(len(facets)):
            have = int((picked.get("facet_best_i", -1) == i).sum())
            if have >= MIN_PER_FACET:
                continue
            need = MIN_PER_FACET - have

            candidates = pool_scored[(pool_scored["facet_best_i"] == i) & (~pool_scored["merge_key"].isin(picked["merge_key"]))]
            if candidates.empty:
                continue
            add = candidates.head(need)

            for _, row in add.iterrows():
                if len(picked) < CAND_TARGET_N:
                    picked = pd.concat([picked, row.to_frame().T], axis=0)
                else:
                    drop_key = weakest_replaceable(picked)
                    if drop_key is None:
                        break
                    picked = picked[picked["merge_key"].astype(str) != drop_key]
                    picked = pd.concat([picked, row.to_frame().T], axis=0)

        picked = picked.drop_duplicates(subset=["merge_key"]).reset_index(drop=True)

    # Final trim (keep all top band)
    if len(picked) > CAND_TARGET_N:
        keep_top = picked[picked["rank"] <= CAND_TOP_N]
        rest = picked[picked["rank"] > CAND_TOP_N].sort_values(rank_col, ascending=False)
        needed = CAND_TARGET_N - len(keep_top)
        picked = pd.concat([keep_top, rest.head(max(0, needed))], axis=0).drop_duplicates(subset=["merge_key"]).reset_index(drop=True)

    return picked


In [14]:
# -----------------------------
# Build candidates for all chapters
# -----------------------------

all_candidates = []

for ch in CHAPTERS:
    cid = ch["chapter_id"]
    bp = blueprints[cid]
    stagea_path = STAGEA_DIR / f"stageA_combined_oa_s2_{cid}.csv"
    if not stagea_path.exists():
        raise RuntimeError(f"Missing {stagea_path}. Run the Stage A fetch cell first.")

    df = pd.read_csv(stagea_path)
    print(f"[{cid}] loaded StageA: {df.shape} | {stagea_path}")

    required_cols = ["merge_key", "title", "abstract"]
    for c in required_cols:
        if c not in df.columns:
            raise RuntimeError(f"Missing required column in StageA CSV ({cid}): {c}")

    df = df.copy()
    df["title"] = df["title"].fillna("")
    df["abstract"] = df["abstract"].fillna("")
    df["doc_text"] = (df["title"].astype(str) + "\n\n" + df["abstract"].astype(str)).str.strip()

    # Citation normalization (per-chapter)
    cites = pd.to_numeric(df.get("citation_count_max", 0), errors="coerce").fillna(0).clip(lower=0)
    df["score_cite"] = np.log1p(cites)
    df["score_cite_norm"] = (df["score_cite"] / df["score_cite"].max()) if df["score_cite"].max() > 0 else 0.0

    # Fit TF-IDF per chapter
    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=TFIDF_NGRAM_RANGE,
        max_features=TFIDF_MAX_FEATURES,
        min_df=TFIDF_MIN_DF,
    )
    X = vectorizer.fit_transform(df["doc_text"])

    # Chapter TF-IDF relevance
    chapter_query_text = build_chapter_query_text(bp)
    df["score_tfidf"] = tfidf_scores(chapter_query_text, vectorizer=vectorizer, X=X)
    df["score_stageC1"] = (1 - CITE_WEIGHT) * df["score_tfidf"] + CITE_WEIGHT * df["score_cite_norm"]

    # Pool via facet-union
    pool = facet_union_pool(bp, df_in=df, vectorizer=vectorizer, X=X, top_per_query=TOP_PER_QUERY)
    print(f"[{cid}] pool size: {len(pool)}")

    # Embedding hybrid scoring within the pool
    pool_scored = score_pool_with_embeddings(bp, pool)
    pool_scored["score_tfidf_norm"] = minmax(pool_scored["score_tfidf"].values)
    pool_scored["score_relevance_hybrid"] = W_EMBED * pool_scored["score_embed_combo"] + W_TFIDF * pool_scored["score_tfidf_norm"]
    pool_scored["score_hybrid_pool"] = (1 - CITE_WEIGHT) * pool_scored["score_relevance_hybrid"] + CITE_WEIGHT * pool_scored["score_cite_norm"]

    # Candidate selection
    cand = build_candidate_set(pool_scored, rank_col="score_hybrid_pool", bp=bp).copy()
    # StageA already contains chapter_id; keep it consistent and put it first.
    cand["chapter_id"] = cid
    if cand.columns[0] != "chapter_id":
        cand = cand[["chapter_id"] + [c for c in cand.columns if c != "chapter_id"]]

    # Keep useful columns (only those present)
    cols_keep = [
        "chapter_id",
        "merge_key", "title", "abstract", "year", "venue", "type", "sources", "doi_norm", "citation_count_max",
        "rank",
        "facet_best_i", "facet_best_query",
        "score_tfidf", "score_stageC1", "score_cite_norm",
        "score_embed_max", "score_embed_mean_top3", "score_embed_combo",
        "score_relevance_hybrid", "score_hybrid_pool",
    ]
    cand_out = cand[[c for c in cols_keep if c in cand.columns]].copy()

    out_path = CAND_DIR / f"{cid}_candidates.csv"
    cand_out.to_csv(out_path, index=False)
    print(f"[{cid}] saved candidates: {out_path} (n={len(cand_out)})")

    all_candidates.append(cand_out)

candidates_all = pd.concat(all_candidates, axis=0).reset_index(drop=True)
combined_path = CAND_DIR / "all_candidates.csv"
candidates_all.to_csv(combined_path, index=False)

print("Saved combined candidates:", combined_path, "n=", len(candidates_all))
print("Embedding usage summary:", embed_totals)

candidates_all.head(5)


[platform_theory] loaded StageA: (4430, 20) | eval_dataset\stageA\stageA_combined_oa_s2_platform_theory.csv
[platform_theory] pool size: 1841
[platform_theory] saved candidates: eval_dataset\candidates\platform_theory_candidates.csv (n=220)
[platform_methodology] loaded StageA: (4295, 20) | eval_dataset\stageA\stageA_combined_oa_s2_platform_methodology.csv
[platform_methodology] pool size: 1741
[platform_methodology] saved candidates: eval_dataset\candidates\platform_methodology_candidates.csv (n=220)
[platform_empirical_case] loaded StageA: (3781, 20) | eval_dataset\stageA\stageA_combined_oa_s2_platform_empirical_case.csv
[platform_empirical_case] pool size: 1396
[platform_empirical_case] saved candidates: eval_dataset\candidates\platform_empirical_case_candidates.csv (n=220)
Saved combined candidates: eval_dataset\candidates\all_candidates.csv n= 660
Embedding usage summary: {'requests': 52, 'input_tokens': 689681, 'cost_usd': 0.013793620000000005}


,chapter_id,merge_key,title,abstract,year,venue,type,sources,doi_norm,citation_count_max,...,facet_best_i,facet_best_query,score_tfidf,score_stageC1,score_cite_norm,score_embed_max,score_embed_mean_top3,score_embed_combo,score_relevance_hybrid,score_hybrid_pool
0,platform_theory,ty:identification of network externalities in ...,Identification of Network Externalities in Mar...,,2002.0,NaN,NaN,semantic_scholar,NaN,10,...,8,empirical identification of network externalities,0.29248,0.284099,0.187721,0.753985,0.616116,0.909083,0.926866,0.867734
1,platform_theory,doi:10.1007/s12525-019-00385-4,Multi-sided platforms,,2019.0,Electronic Markets,NaN,semantic_scholar,10.1007/s12525-019-00385-4,37,...,1,multi-sided platforms value creation mechanisms,0.237598,0.241372,0.284771,0.773966,0.687186,0.967444,0.889813,0.84141
2,platform_theory,doi:10.2139/ssrn.975897,Two-Sided Markets and Price Competition With M...,,2004.0,NaN,NaN,semantic_scholar,10.2139/ssrn.975897,109,...,4,multi-homing switching costs market share effects,0.277464,0.284705,0.367981,0.66954,0.664064,0.84581,0.869184,0.829088
3,platform_theory,doi:10.1111/j.1467-6451.2010.00426.x,Tying in Two-Sided Markets with Multi-Homing,This paper analyzes the effects of tying arran...,2007.0,Social Science Research Network,NaN,semantic_scholar,10.1111/j.1467-6451.2010.00426.x,325,...,4,multi-homing switching costs market share effects,0.266002,0.280964,0.453032,0.657591,0.636438,0.818741,0.837891,0.807102
4,platform_theory,doi:https://doi.org/10.1257/jep.8.2.133,Network Externality: An Uncommon Tragedy,Economists have defined ‘network externality’ ...,1994.0,The Journal of Economic Perspectives,article,openalex,https://doi.org/10.1257/jep.8.2.133,1083,...,8,empirical identification of network externalities,0.249022,0.272868,0.547094,0.690764,0.609907,0.839545,0.828075,0.805597


In [15]:
# -----------------------------
# Labeling with an LLM (cached + cost tracked)
# -----------------------------

LABEL_INSTRUCTIONS_VERSION = "v1_include_maybe_exclude"

class CandidateLabel(BaseModel):
    label: Literal["include", "maybe", "exclude"] = Field(..., description="Primary label")
    confidence: int = Field(..., ge=0, le=100)
    must_cover_hit_indices: List[int] = Field(..., description="0-based indices into MUST_COVER")
    must_avoid_violation_indices: List[int] = Field(..., description="0-based indices into MUST_AVOID")
    tags: List[str] = Field(..., description="0–3 tags from: scope_violation, wrong_level, too_broad, low_signal, wrong_domain")
    short_reason: str = Field(..., description="<= 20 words")

LABELER_INSTRUCTIONS = (
    "You label academic sources for a thesis chapter using ONLY the provided CHAPTER_BLUEPRINT.\n\n"
    "Decide if the candidate is a GOOD source for writing this chapter.\n\n"
    "LABEL DEFINITIONS:\n"
    "- include: strong fit; supports MUST_COVER; no MUST_AVOID violations.\n"
    "- maybe: partial fit; could be used, but missing key MUST_COVER or somewhat broad/unclear.\n"
    "- exclude: off-scope OR violates MUST_AVOID OR wrong level for this chapter.\n\n"
    "RULES:\n"
    "- Use evidence in title/abstract only.\n"
    "- If abstract is missing/very short: low confidence + tag low_signal.\n"
    "- tags must be from: scope_violation, wrong_level, too_broad, low_signal, wrong_domain.\n"
    "- short_reason must be <= 20 words.\n\n"
    "Return ONLY the structured output.\n"
)

label_agent_pass1 = Agent(
    name="Dataset Labeler (pass1)",
    model=LABEL_MODEL_PASS1,
    model_settings=ModelSettings(verbosity="low"),
    instructions=LABELER_INSTRUCTIONS,
    output_type=CandidateLabel,
)

label_agent_pass2 = Agent(
    name="Dataset Labeler (pass2)",
    model=LABEL_MODEL_PASS2,
    model_settings=ModelSettings(verbosity="low"),
    instructions=LABELER_INSTRUCTIONS,
    output_type=CandidateLabel,
)

def fmt_indexed(items: List[str]) -> str:
    return "\n".join([f"{i}. {x}" for i, x in enumerate(items or [])]) if items else "(none)"

def build_label_prompt(bp: dict, row: pd.Series) -> str:
    must_cover = bp.get("must_cover") or []
    must_avoid = bp.get("must_avoid") or []

    title = str(row.get("title") or "").strip()
    abstract = str(row.get("abstract") or "").strip().replace("\n", " ")
    abstract = abstract[:MAX_ABS_CHARS_FOR_LABEL]

    year = row.get("year")
    venue = row.get("venue")
    typ = row.get("type")
    cites = row.get("citation_count_max")

    return (
        "CHAPTER_BLUEPRINT\n"
        f"SCOPE_STATEMENT: {bp.get('scope_statement','')}\n"
        "MUST_COVER (0-based):\n" + fmt_indexed(must_cover) + "\n"
        "MUST_AVOID (0-based):\n" + fmt_indexed(must_avoid) + "\n"
        f"SCORING_GUIDANCE: {bp.get('scoring_guidance','')}\n"
        "\n"
        "CANDIDATE\n"
        f"TITLE: {title}\n"
        f"YEAR: {year}\n"
        f"TYPE: {typ}\n"
        f"VENUE: {venue}\n"
        f"CITATIONS: {cites}\n"
        f"ABSTRACT: {abstract}\n"
    )

def cache_path(prompt: str, model: str) -> Path:
    blob = LABEL_INSTRUCTIONS_VERSION + "\n" + model + "\n" + prompt
    h = hashlib.sha1(blob.encode("utf-8")).hexdigest()
    return LABEL_CACHE_DIR / f"{h}.json"

async def label_one(idx: int, row: pd.Series, agent: Agent, model: str, max_retries: int = 6):
    bp = blueprints[row["chapter_id"]]
    prompt = build_label_prompt(bp, row)
    cp = cache_path(prompt, model)

    if cp.exists():
        out = json.loads(cp.read_text(encoding="utf-8"))
        usage0 = {"requests": 0, "input_tokens": 0, "cached_input_tokens": 0, "output_tokens": 0, "cost_usd": 0.0}
        return idx, out, usage0

    backoff = 1.0
    for attempt in range(1, max_retries + 1):
        try:
            res = await Runner.run(agent, prompt)
            out = res.final_output.model_dump()
            cp.write_text(json.dumps(out, ensure_ascii=False), encoding="utf-8")

            usage = res.context_wrapper.usage
            usage_dict = cost_from_usage(usage, model=model)
            return idx, out, usage_dict
        except Exception:
            if attempt == max_retries:
                raise
            await asyncio.sleep(backoff + 0.15 * attempt)
            backoff *= 2

async def label_many(df_in: pd.DataFrame, which_pass: str, agent: Agent, model: str, concurrency: int, budget_state: Optional[dict] = None):
    sem = asyncio.Semaphore(concurrency)

    async def wrapped(idx, row):
        async with sem:
            return await label_one(idx, row, agent=agent, model=model)

    tasks = [asyncio.create_task(wrapped(i, row)) for i, row in df_in.iterrows()]

    totals = {"requests": 0, "input_tokens": 0, "cached_input_tokens": 0, "output_tokens": 0, "cost_usd": 0.0}
    results = {}

    t0 = time.monotonic()
    pbar = tqdm(asyncio.as_completed(tasks), total=len(tasks), desc=f"label {which_pass} ({model})")

    for fut in pbar:
        idx, out, u = await fut
        results[idx] = out

        for k in totals:
            totals[k] += u.get(k, 0)

        elapsed = time.monotonic() - t0
        pbar.set_postfix({
            "req": totals["requests"],
            "in_tok": totals["input_tokens"],
            "cached": totals["cached_input_tokens"],
            "out_tok": totals["output_tokens"],
            "cost_$": f"{totals['cost_usd']:.4f}",
            "sec": f"{elapsed:.0f}",
        })

        if budget_state is not None:
            budget_state["cost_usd"] = float(budget_state.get("cost_usd", 0.0)) + float(u.get("cost_usd", 0.0))
            if STOP_ON_BUDGET and budget_state["cost_usd"] > BUDGET_USD:
                raise RuntimeError(f"Budget exceeded (total): {budget_state['cost_usd']:.2f} > {BUDGET_USD:.2f}. Stop.")

    df_out = pd.DataFrame([results[i] for i in range(len(df_in))])
    return df_out, totals

print("Labeling utilities ready")


Labeling utilities ready


In [16]:
# -----------------------------
# Run labeling (pass1 + optional pass2)
# -----------------------------

budget_state = {"cost_usd": 0.0}
budget_state["cost_usd"] += float(bp_totals.get("cost_usd", 0.0))
budget_state["cost_usd"] += float(embed_totals.get("cost_usd", 0.0))
print(f"Starting cost (blueprints+embeddings): ${budget_state['cost_usd']:.4f} / ${BUDGET_USD:.2f}")

# Pass 1: label everything
labels1, totals1 = await label_many(
    candidates_all,
    which_pass="pass1",
    agent=label_agent_pass1,
    model=LABEL_MODEL_PASS1,
    concurrency=LABEL_CONCURRENCY,
    budget_state=budget_state,
)

print("\n=== Pass1 totals ===")
print(totals1)
print(f"Total cost so far: ${budget_state['cost_usd']:.4f} / ${BUDGET_USD:.2f}")

df_l1 = pd.concat([candidates_all.reset_index(drop=True), labels1.add_prefix("p1_")], axis=1)
out1 = LABEL_DIR / "labels_pass1.csv"
df_l1.to_csv(out1, index=False)
print("Saved:", out1)

# Pass 2: relabel uncertain + audit
if USE_PASS2:
    redo = (df_l1["p1_confidence"] < PASS2_CONFIDENCE_LT)
    if PASS2_REDO_MAYBE:
        redo = redo | (df_l1["p1_label"] == "maybe")

    audit_idx = []
    for cid, g in df_l1.groupby("chapter_id"):
        n = min(PASS2_RANDOM_AUDIT_PER_CHAPTER, len(g))
        if n > 0:
            audit_idx.extend(g.sample(n=n, random_state=SEED).index.tolist())

    redo_idx = sorted(set(df_l1[redo].index.tolist() + audit_idx))
    df_redo = df_l1.loc[redo_idx, candidates_all.columns].reset_index(drop=True)

    print("\nPass2 relabel count:", len(df_redo))

    if len(df_redo) > 0:
        labels2, totals2 = await label_many(
            df_redo,
            which_pass="pass2",
            agent=label_agent_pass2,
            model=LABEL_MODEL_PASS2,
            concurrency=max(10, LABEL_CONCURRENCY // 2),
            budget_state=budget_state,
        )
        print("\n=== Pass2 totals ===")
        print(totals2)
        print(f"Total cost so far: ${budget_state['cost_usd']:.4f} / ${BUDGET_USD:.2f}")

        df_l2 = pd.concat([df_redo.reset_index(drop=True), labels2.add_prefix("p2_")], axis=1)
        out2 = LABEL_DIR / "labels_pass2_subset.csv"
        df_l2.to_csv(out2, index=False)
        print("Saved:", out2)

        # Merge back into df_l1
        df_final = df_l1.copy()
        tmp = df_l2.copy()
        tmp["_orig_idx"] = redo_idx
        tmp = tmp.set_index("_orig_idx")
        for col in [c for c in tmp.columns if c.startswith("p2_")]:
            df_final.loc[tmp.index, col] = tmp[col]

        df_final["final_label"] = df_final["p2_label"].fillna(df_final["p1_label"])
        df_final["final_confidence"] = df_final["p2_confidence"].fillna(df_final["p1_confidence"])
    else:
        df_final = df_l1.copy()
        df_final["final_label"] = df_final["p1_label"]
        df_final["final_confidence"] = df_final["p1_confidence"]
else:
    df_final = df_l1.copy()
    df_final["final_label"] = df_final["p1_label"]
    df_final["final_confidence"] = df_final["p1_confidence"]

# Save combined labeled dataset
labeled_path = OUT_DIR / "labeled_dataset.csv"
df_final.to_csv(labeled_path, index=False)
print("\nSaved labeled dataset:", labeled_path)
print(f"Final estimated total cost: ${budget_state['cost_usd']:.4f} / ${BUDGET_USD:.2f}")

print("\nLabel counts by chapter:")
display(df_final.groupby(["chapter_id", "final_label"]).size().unstack(fill_value=0))

print("\nAvg confidence by chapter:")
display(df_final.groupby("chapter_id")["final_confidence"].mean().round(1))

# Quick look
df_final[["chapter_id", "final_label", "final_confidence", "score_hybrid_pool", "title"]].head(20)


Starting cost (blueprints+embeddings): $0.0298 / $2.00


label pass1 (gpt-5-nano):   0%|          | 0/660 [00:00<?, ?it/s]


=== Pass1 totals ===
{'requests': 660, 'input_tokens': 518103, 'cached_input_tokens': 0, 'output_tokens': 1243860, 'cost_usd': 0.5234491500000004}
Total cost so far: $0.5532 / $2.00
Saved: eval_dataset\labels\labels_pass1.csv

Pass2 relabel count: 515


label pass2 (gpt-5-mini):   0%|          | 0/515 [00:00<?, ?it/s]


=== Pass2 totals ===
{'requests': 515, 'input_tokens': 391316, 'cached_input_tokens': 0, 'output_tokens': 397284, 'cost_usd': 0.892397}
Total cost so far: $1.4456 / $2.00
Saved: eval_dataset\labels\labels_pass2_subset.csv

Saved labeled dataset: eval_dataset\labeled_dataset.csv
Final estimated total cost: $1.4456 / $2.00

Label counts by chapter:


final_label,exclude,include,maybe
chapter_id,,,
platform_empirical_case,139,4,77
platform_methodology,62,4,154
platform_theory,84,33,103



Avg confidence by chapter:


chapter_id
platform_empirical_case    67.0
platform_methodology       61.4
platform_theory            62.4
Name: final_confidence, dtype: float64

,chapter_id,final_label,final_confidence,score_hybrid_pool,title
0,platform_theory,maybe,40.0,0.867734,Identification of Network Externalities in Mar...
1,platform_theory,maybe,40.0,0.84141,Multi-sided platforms
2,platform_theory,include,45.0,0.829088,Two-Sided Markets and Price Competition With M...
3,platform_theory,include,80.0,0.807102,Tying in Two-Sided Markets with Multi-Homing
4,platform_theory,maybe,75.0,0.805597,Network Externality: An Uncommon Tragedy
5,platform_theory,include,60.0,0.801414,Platform competition with partial multi-homing...
6,platform_theory,maybe,40.0,0.800164,Switching Costs in Two‐Sided Markets
7,platform_theory,maybe,40.0,0.798085,Competition in Two-Sided Markets with Common N...
8,platform_theory,include,40.0,0.791688,Two-sided markets: a progress report
9,platform_theory,maybe,90.0,0.789155,Modeling the Dynamics of Network Entry and Com...


## How to run

1) Set env vars:
- `OPENAI_API_KEY` (required)
- `OPENALEX_API_KEY` (recommended)
- `S2_API_KEY` (recommended)

2) Run cells top-to-bottom.

3) Inspect outputs:
- `eval_dataset/blueprints/*.json`
- `eval_dataset/fetch/openalex_*.csv` and `eval_dataset/fetch/semantic_scholar_*.csv`
- `eval_dataset/stageA/stageA_combined_oa_s2_*.csv` (and `eval_dataset/stageA/stageA_all_chapters.csv`)
- `eval_dataset/candidates/*.csv`
- `eval_dataset/labels/labels_pass1.csv` (+ pass2 subset if enabled)
- `eval_dataset/labeled_dataset.csv`

Notes:
- Re-running is cheap because of caches: `.embed_cache/`, `.llm_label_cache_v1/`, and `eval_dataset/fetch/s2_cache/`.
- Main knobs are in `eval_config`: `FETCH_*`, `OA_*`, `S2_*`, `CAND_TARGET_N`, `TOP_PER_QUERY`, `MAX_ABS_CHARS_FOR_LABEL`, `USE_PASS2`, `PASS2_*`, `BUDGET_USD`.
